# 02 — Chia tập, chuẩn hóa và tạo cửa sổ

Chạy độc lập trong kernel mới, không dựa vào biến của notebook 01. Giai đoạn này chỉ
chuẩn bị dữ liệu, kiểm tra invariants và lưu cấu hình đề xuất; **không gọi model.fit,
không chọn ngưỡng thực tế, không đánh giá mô hình**.

Các số liệu audit và window count trong output là kết quả chạy thật. Kiểm tra hàm
scoring bằng ví dụ giả lập được gắn nhãn rõ, không phải reconstruction của LSTM.

Môi trường bắt buộc: `D:\LSTM\.venv`, kernel **Python (LSTM .venv)**.
Cell đầu in và kiểm tra `sys.executable` trước khi xử lý dữ liệu; không dùng `drone_rl`.


In [1]:
from pathlib import Path
import sys, json, platform
EXPECTED_PYTHON = Path(r'D:\LSTM\.venv\Scripts\python.exe')
print('sys.executable:', sys.executable)
assert Path(sys.executable).resolve() == EXPECTED_PYTHON.resolve(), 'Chon kernel Python (LSTM .venv)'
assert sys.prefix != sys.base_prefix, 'Virtual environment required'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'utils.py').is_file())
sys.path.insert(0, str(ROOT))
from src.utils import (download_nab, load_series, audit_series, NORMAL, TEST,
                       chronological_splits, make_windows, endpoint_mse, normal_threshold)
plt.rcParams.update({'figure.figsize': (12, 3), 'axes.grid': True})
print('Python:', platform.python_version(), '| Project:', ROOT)

from sklearn.preprocessing import StandardScaler
import sklearn
raw, manifest = download_nab(ROOT)
normal = load_series(raw, 'normal.csv', NORMAL)
test = load_series(raw, 'test.csv', TEST)
print('NumPy:', np.__version__, '| pandas:', pd.__version__, '| sklearn:', sklearn.__version__)

sys.executable: D:\LSTM\.venv\Scripts\python.exe


Python: 3.12.10 | Project: D:\LSTM\anomaly-lstm


NumPy: 1.26.4 | pandas: 2.2.2 | sklearn: 1.5.1


## Giao thức thực nghiệm đề xuất — tách vai trò trước windowing

| Tập | Nguồn, hàng gốc (0-based, bao gồm hai đầu) | Vai trò về sau |
|---|---|---|
| train | small_noise, 0–2303, 8 ngày | Fit scaler và trọng số mô hình; toàn normal |
| validation | small_noise, 2304–3167, 3 ngày | Normal reconstruction loss cho early stopping/checkpoint; không chọn ngưỡng |
| calibration | small_noise, 3168–4031, 3 ngày | Chọn ngưỡng chỉ sau khi khóa mô hình; không cập nhật trọng số/scaler |
| test | jumpsup, toàn chuỗi riêng | Một lần đánh giá cuối với nhãn nguồn đã chuyển đổi |

Split là **do dự án đặt ra**, không phải split train/test chính thức của NAB.
Không random split trên các cửa sổ chồng lấn. Các tập normal nối tiếp theo thời gian;
test là nguồn độc lập có cùng trục timestamp, không được hiểu là một tương lai lịch
muộn hơn train. Không nối hai nguồn hay lọc bỏ anomaly rồi ghép các đoạn còn lại.
Không cần embargo để ngăn chia sẻ quan sát khi split trước windowing; tự tương quan
và mẫu ngày tương tự giữa các tập vẫn còn, nên không diễn giải chúng là IID.

In [2]:
parts = chronological_splits(normal, test)
split_summary = pd.DataFrame([
    {'split': name, 'source': frame.source_id.iloc[0], 'rows': len(frame),
     'first_row': int(frame.source_row.iloc[0]), 'last_row': int(frame.source_row.iloc[-1]),
     'start': str(frame.timestamp.iloc[0]), 'end': str(frame.timestamp.iloc[-1]),
     'normal': int(frame.label.eq(0).sum()), 'anomaly': int(frame.label.eq(1).sum())}
    for name, frame in parts.items()
])
display(split_summary)
keys = {name: set(zip(f.source_id, f.source_row)) for name, f in parts.items()}
for i, left in enumerate(parts):
    for right in list(parts)[i+1:]:
        assert keys[left].isdisjoint(keys[right]), (left, right)
assert set.union(*(keys[k] for k in ['train', 'validation', 'calibration'])) == set(zip(normal.source_id, normal.source_row))
assert parts['train'].timestamp.max() < parts['validation'].timestamp.min()
assert parts['validation'].timestamp.max() < parts['calibration'].timestamp.min()
assert all(parts[k].label.eq(0).all() for k in ['train', 'validation', 'calibration'])
assert keys['test'] == set(zip(test.source_id, test.source_row))
print('PASS: disjoint (source_id, source_row), complete partition, chronology, normal-only development')

,split,source,rows,first_row,last_row,start,end,normal,anomaly
0,train,artificialNoAnomaly/art_daily_small_noise.csv,2304,0,2303,2014-04-01 00:00:00,2014-04-08 23:55:00,2304,0
1,validation,artificialNoAnomaly/art_daily_small_noise.csv,864,2304,3167,2014-04-09 00:00:00,2014-04-11 23:55:00,864,0
2,calibration,artificialNoAnomaly/art_daily_small_noise.csv,864,3168,4031,2014-04-12 00:00:00,2014-04-14 23:55:00,864,0
3,test,artificialWithAnomaly/art_daily_jumpsup.csv,4032,0,4031,2014-04-01 00:00:00,2014-04-14 23:55:00,3629,403


PASS: disjoint (source_id, source_row), complete partition, chronology, normal-only development


## Chuẩn hóa train-only

Một `StandardScaler` cho biến value, fit trên **2.304 điểm train gốc**, trước khi
nhân bản điểm do cửa sổ chồng lấn. Dùng đúng mean/scale đó transform mọi tập, không
fit lại; không chuẩn hóa từng cửa sổ, vì cách đó có thể xóa độ lệch mức cần phát hiện.
Không clip giá trị test ngoài dải train. Lưu tham số JSON thay vì pickle.

In [3]:
scaler = StandardScaler()
scaler.fit(parts['train'][['value']].to_numpy())
mean_before, scale_before = scaler.mean_.copy(), scaler.scale_.copy()
scaled = {name: scaler.transform(frame[['value']].to_numpy()) for name, frame in parts.items()}
train_values = parts['train'].value.to_numpy()
assert int(scaler.n_samples_seen_) == len(parts['train'])
np.testing.assert_allclose(scaler.mean_, [train_values.mean()])
np.testing.assert_allclose(scaler.scale_, [train_values.std(ddof=0)])
np.testing.assert_allclose(scaled['train'].mean(axis=0), 0, atol=1e-12)
np.testing.assert_allclose(scaled['train'].std(axis=0), 1, atol=1e-12)
for name, values in scaled.items():
    np.testing.assert_allclose(values, (parts[name][['value']].to_numpy() - mean_before) / scale_before)
    np.testing.assert_allclose(scaler.inverse_transform(values), parts[name][['value']].to_numpy())
np.testing.assert_array_equal(scaler.mean_, mean_before)
np.testing.assert_array_equal(scaler.scale_, scale_before)
# Counterfactual: transforming extreme held-out values cannot mutate fitted state.
scaler.transform(np.array([[1e9], [-1e9]]))
np.testing.assert_array_equal(scaler.mean_, mean_before)
np.testing.assert_array_equal(scaler.scale_, scale_before)
display(pd.DataFrame({'mean_train': scaler.mean_, 'std_train_ddof0': scaler.scale_,
                      'fit_samples': int(scaler.n_samples_seen_)}, index=['value']))
display(pd.DataFrame({name: {'mean_scaled': values.mean(), 'std_scaled': values.std(),
                            'min_scaled': values.min(), 'max_scaled': values.max()}
                      for name, values in scaled.items()}).T)
print('PASS: train-only scaler, correct transforms, inverse round trip, immutable state')

,mean_train,std_train_ddof0,fit_samples
value,42.435958,28.034265,2304


,mean_scaled,std_scaled,min_scaled,max_scaled
train,7.093092e-17,1.000000,-0.871512,1.621932
validation,2.209239e-03,1.003180,-0.871612,1.624447
calibration,-1.810477e-03,1.003364,-0.871589,1.623404
test,7.342073e-02,1.156743,-0.871610,4.370064


PASS: train-only scaler, correct transforms, inverse round trip, immutable state


## Sliding windows và ánh xạ về nhãn

Đề xuất `length=288` (một ngày, 5 phút/bước), `stride=1`, một biến. Đây là lựa chọn
theo chu kỳ lấy mẫu và giả định mùa vụ, chưa tối ưu bằng nhãn. Mỗi lần gọi hàm chỉ
nhận đúng một split của một nguồn; kiểm tra cả timestamp liên tục và chỉ số hàng gốc.
Một cửa sổ chứa các hàng `[start, end]`, với `end=start+287`.

Lưu hai nhãn: `label_endpoint` là nhãn điểm cuối để đánh giá chính; `label_any` chỉ
là nhãn chẩn đoán cửa sổ có ít nhất một điểm trong khoảng sự kiện. Không dùng
`label_any` thay nhãn điểm vì sẽ nới vùng dương thêm 287 bước.
Mỗi tập bỏ 287 điểm đầu khỏi tập điểm có score để có đủ lịch sử riêng trong tập;
không mượn context từ tập trước. Không zero-pad hoặc gán score=0 cho điểm không đủ context.

In [4]:
LENGTH, STRIDE = 288, 1
windows, maps = {}, {}
for name, frame in parts.items():
    windows[name], maps[name] = make_windows(frame, scaled[name], length=LENGTH, stride=STRIDE)
window_summary = pd.DataFrame([
    {'split': name, 'shape': str(windows[name].shape),
     'scored_points': len(m), 'unscored_prefix': LENGTH-1,
     'endpoint_positive': int(m.label_endpoint.sum()), 'any_positive': int(m.label_any.sum())}
    for name, m in maps.items()
])
display(window_summary)
display(pd.concat([m.iloc[[0, -1]] for m in maps.values()], ignore_index=True))

,split,shape,scored_points,unscored_prefix,endpoint_positive,any_positive
0,train,"(2017, 288, 1)",2017,287,0,0
1,validation,"(577, 288, 1)",577,287,0,0
2,calibration,"(577, 288, 1)",577,287,0,0
3,test,"(3745, 288, 1)",3745,287,403,690


,window_id,split,source_id,local_start,local_end,source_start,source_end,timestamp_start,timestamp_end,label_endpoint,label_any
0,0,train,artificialNoAnomaly/art_daily_small_noise.csv,0,287,0,287,2014-04-01,2014-04-01 23:55:00,0,0
1,2016,train,artificialNoAnomaly/art_daily_small_noise.csv,2016,2303,2016,2303,2014-04-08,2014-04-08 23:55:00,0,0
2,0,validation,artificialNoAnomaly/art_daily_small_noise.csv,0,287,2304,2591,2014-04-09,2014-04-09 23:55:00,0,0
3,576,validation,artificialNoAnomaly/art_daily_small_noise.csv,576,863,2880,3167,2014-04-11,2014-04-11 23:55:00,0,0
4,0,calibration,artificialNoAnomaly/art_daily_small_noise.csv,0,287,3168,3455,2014-04-12,2014-04-12 23:55:00,0,0
5,576,calibration,artificialNoAnomaly/art_daily_small_noise.csv,576,863,3744,4031,2014-04-14,2014-04-14 23:55:00,0,0
6,0,test,artificialWithAnomaly/art_daily_jumpsup.csv,0,287,0,287,2014-04-01,2014-04-01 23:55:00,0,0
7,3744,test,artificialWithAnomaly/art_daily_jumpsup.csv,3744,4031,3744,4031,2014-04-14,2014-04-14 23:55:00,0,0


In [5]:
checks = {}
for name, frame in parts.items():
    x, m = windows[name], maps[name]
    assert x.shape == (len(frame)-LENGTH+1, LENGTH, 1)
    assert x.dtype == np.float32 and np.isfinite(x).all()
    assert m.source_id.eq(frame.source_id.iloc[0]).all() and m.split.eq(name).all()
    np.testing.assert_array_equal(m.local_start, np.arange(len(m)))
    np.testing.assert_array_equal(m.local_end, m.local_start + LENGTH-1)
    assert m.source_start.min() == frame.source_row.min()
    assert m.source_end.max() == frame.source_row.max()
    assert (m.source_end - m.source_start).eq(LENGTH-1).all()
    assert (m.timestamp_end - m.timestamp_start).eq(pd.Timedelta(minutes=5*(LENGTH-1))).all()
    # Exhaustively compare every saved window, label and mapping to original rows.
    for j, row in m.iterrows():
        a, b = int(row.local_start), int(row.local_end)
        np.testing.assert_array_equal(x[j], scaled[name][a:b+1].astype(np.float32))
        original = frame.iloc[a:b+1]
        assert original.source_row.iloc[0] == row.source_start
        assert original.source_row.iloc[-1] == row.source_end
        assert original.timestamp.iloc[0] == row.timestamp_start
        assert original.timestamp.iloc[-1] == row.timestamp_end
        assert original.label.iloc[-1] == row.label_endpoint
        assert original.label.max() == row.label_any
    if name != 'test':
        assert not m.label_endpoint.any() and not m.label_any.any()
    checks[name] = 'PASS: all windows/indices/timestamps/labels/values checked'
display(pd.Series(checks, name='exhaustive_checks'))

# Check that the actual raw observations used by windows never overlap across splits.
used = {name: {(f.source_id.iloc[0], i) for i in range(int(m.source_start.min()), int(m.source_end.max())+1)}
        for (name, f), m in zip(parts.items(), maps.values())}
for i, left in enumerate(used):
    for right in list(used)[i+1:]:
        assert used[left].isdisjoint(used[right])
assert maps['test'].label_endpoint.sum() == test.label.sum()  # no event lost in warm-up
assert len(test) - len(maps['test']) == LENGTH-1
print('PASS: no cross-boundary raw observation reuse; test anomaly coverage preserved')

train          PASS: all windows/indices/timestamps/labels/va...
validation     PASS: all windows/indices/timestamps/labels/va...
calibration    PASS: all windows/indices/timestamps/labels/va...
test           PASS: all windows/indices/timestamps/labels/va...
Name: exhaustive_checks, dtype: object

PASS: no cross-boundary raw observation reuse; test anomaly coverage preserved


## Kiểm tra cơ chế từ chối đầu vào sai

Các ví dụ sau cố tình trộn tập, trộn nguồn, tạo gap hoặc bỏ hàng. Hàm phải từ chối
thay vì âm thầm tạo cửa sổ sai. Scoring và ngưỡng chỉ kiểm tra với số giả lập.

In [6]:
def expect_value_error(call):
    try:
        call()
    except ValueError:
        return True
    raise AssertionError('Expected rejection')

mixed_split = pd.concat([parts['train'], parts['validation']], ignore_index=True)
mixed_source = pd.concat([parts['train'], parts['test']], ignore_index=True).assign(split='train')
gap = parts['train'].drop(parts['train'].index[100]).copy()
bad_row = parts['train'].copy()
bad_row.loc[bad_row.index[100], 'source_row'] += 2
for f in [mixed_split, mixed_source, gap, bad_row]:
    assert expect_value_error(lambda f=f: make_windows(f, f[['value']].to_numpy()))
assert expect_value_error(lambda: make_windows(parts['train'], scaled['train'], length=99999))
assert expect_value_error(lambda: make_windows(parts['train'], scaled['train'], stride=0))

# Synthetic arithmetic checks only: no model outputs and no real calibration threshold.
toy = np.zeros((2, 3, 1))
toy_reconstruction = np.array([[[9.], [9.], [2.]], [[9.], [9.], [3.]]])
np.testing.assert_array_equal(endpoint_mse(toy, toy_reconstruction), [4., 9.])
assert normal_threshold(np.arange(100.), .99) == 99.
assert expect_value_error(lambda: normal_threshold(np.array([np.nan])))
checks['guards_and_scoring'] = 'PASS: mixed boundaries/gaps rejected; synthetic score and quantile verified'
print(checks['guards_and_scoring'])

PASS: mixed boundaries/gaps rejected; synthetic score and quantile verified


## Đề xuất baseline, LSTM và đánh giá — chờ review, chưa huấn luyện

**Baseline:** (1) tái tạo hằng bằng mean train, giúp phát hiện liệu mô hình phức tạp
có hơn mức tối thiểu; (2) seasonal median theo 288 slot trong ngày, fit chỉ 8 ngày train,
tái tạo mỗi timestep theo slot timestamp. Baseline mùa vụ phù hợp giả định chu kỳ và
không lấy mẫu test làm template. Cả hai dùng cùng scaler, điểm cuối cửa sổ, coverage
và quy tắc ngưỡng như LSTM. Không tính kết quả baseline trong lượt này.

**LSTM khởi đầu:** Input `(288,1)` → LSTM encoder 32 units → vector latent →
RepeatVector(288) → LSTM decoder 32 units (`return_sequences=True`) → Dense(1)
tại mỗi bước. Tái tạo cùng chiều, không teacher forcing, loss MSE trên toàn cửa sổ.
Adam `lr=1e-3`, batch 32, tối đa 50 epoch, seed 42; `shuffle=False`, early stopping
theo validation MSE, patience 5, restore best weights. Đây là đề xuất đơn giản của
dự án, khác decoder đảo chiều và điểm Mahalanobis trong
[EncDec-AD](https://arxiv.org/html/1607.00148v2). Chưa tạo model/weights/backend training.

**Điểm chính:** `s(t) = mean_feature((x[t] - x_hat[t | t-287:t])**2)`, tức lỗi
tại điểm cuối của cửa sổ kết thúc ở t. Model thấy dữ liệu tới t, không dùng dữ liệu sau t.
Training loss toàn cửa sổ và detection score điểm cuối là hai đại lượng khác nhau.
Không trung bình reconstruction từ các cửa sổ kết thúc trong tương lai. Nếu sau này
báo cáo thêm window-MSE/label-any thì phải ghi thành thí nghiệm riêng.

**Ngưỡng:** khóa checkpoint từ validation trước; lấy score của calibration normal,
`tau = quantile(scores_cal, 0.99, method='higher')`, dự đoán `score > tau`.
Mỗi phương pháp có ngưỡng riêng theo cùng quy tắc. Không dùng nhãn test hoặc phân phối
score test để đặt quantile, chọn anomaly ratio hoặc tìm best F1. Không refit bằng
validation/calibration sau khi đã dùng chúng. Chỉ có 3 ngày calibration, các điểm chồng
lấn tương quan cao; 1% là tỷ lệ đuôi danh nghĩa, không bảo đảm FPR trên test hay dữ liệu thật.

**Đánh giá sau này:** trên đúng endpoint được ánh xạ (bỏ prefix 287 điểm), dùng nhãn
`label_endpoint`; báo confusion matrix, precision, recall, F1, Average Precision (AP) và ROC-AUC, cùng tỷ lệ dương và số điểm có score. Không dùng point
adjustment, không tô cả sự kiện dương khi chỉ hit một điểm, không chọn ngưỡng từ đường
PR test. Với tập chỉ normal báo FPR thay vì ROC-AUC không xác định. Không so metric điểm
này với NAB score. Chỉ một sự kiện nhân tạo nên độ tin cậy/generalization còn hạn chế.

In [7]:
plan = {
    'status': 'proposal_for_review_no_training', 'seed': 42,
    'dataset_commit': manifest['commit'], 'normal_source': NORMAL, 'test_source': TEST,
    'split_rows': {k: [int(f.source_row.iloc[0]), int(f.source_row.iloc[-1])] for k, f in parts.items()},
    'window_length': LENGTH, 'stride': STRIDE, 'feature_names': ['value'],
    'label_policy': 'inclusive NAB combined_windows, endpoint primary, any diagnostic',
    'score': 'endpoint feature-mean squared reconstruction error',
    'threshold': {'source': 'calibration normal only', 'quantile': .99, 'method': 'higher', 'comparison': '>'},
    'baselines': ['train mean reconstruction', 'train slot-of-day median reconstruction'],
    'lstm': {'encoder_units': 32, 'decoder_units': 32, 'repeat_vector': LENGTH,
             'output': 'Dense(1) per timestep', 'reconstruction_order': 'forward',
             'loss': 'full-window MSE', 'optimizer': 'Adam', 'learning_rate': .001,
             'batch_size': 32, 'max_epochs': 50, 'shuffle': False,
             'early_stopping': {'monitor': 'validation MSE', 'patience': 5, 'restore_best_weights': True}},
    'metrics_later': ['confusion matrix', 'precision', 'recall', 'F1', 'average precision', 'ROC-AUC'],
    'point_adjustment': False, 'test_used_for': 'data audit only in stage 1; final evaluation later',
}
display(plan)

{'status': 'proposal_for_review_no_training',
 'seed': 42,
 'dataset_commit': 'ea702d75cc2258d9d7dd35ca8e5e2539d71f3140',
 'normal_source': 'artificialNoAnomaly/art_daily_small_noise.csv',
 'test_source': 'artificialWithAnomaly/art_daily_jumpsup.csv',
 'split_rows': {'train': [0, 2303],
  'validation': [2304, 3167],
  'calibration': [3168, 4031],
  'test': [0, 4031]},
 'window_length': 288,
 'stride': 1,
 'feature_names': ['value'],
 'label_policy': 'inclusive NAB combined_windows, endpoint primary, any diagnostic',
 'score': 'endpoint feature-mean squared reconstruction error',
 'threshold': {'source': 'calibration normal only',
  'quantile': 0.99,
  'method': 'higher',
  'comparison': '>'},
 'baselines': ['train mean reconstruction',
  'train slot-of-day median reconstruction'],
 'lstm': {'encoder_units': 32,
  'decoder_units': 32,
  'repeat_vector': 288,
  'output': 'Dense(1) per timestep',
  'reconstruction_order': 'forward',
  'loss': 'full-window MSE',
  'optimizer': 'Adam',
  'l

## Lưu artifact và kiểm tra đọc lại

`data/processed/windows.npz`: tensor từng split + hai nhãn tương ứng;
`window_mapping.csv`: nguồn, split, hàng gốc, chỉ số cục bộ, timestamp, nhãn mỗi cửa sổ;
`split_rows.csv`: từng hàng gốc và vai trò; `scaler.json`: mean/scale train và provenance.
`results/experiment_plan.json` là cấu hình đề xuất; `preprocessing_checks.json` lưu
kết quả kiểm tra. Artifact sinh lại sẽ được ghi lại khi chạy notebook, raw nguồn được
giữ và kiểm hash. Không có checkpoint hay ngưỡng thực tế ở giai đoạn này.

In [8]:
processed = ROOT / 'data' / 'processed'
processed.mkdir(parents=True, exist_ok=True)
(ROOT / 'results').mkdir(exist_ok=True)
payload = {}
for name in parts:
    payload[f'X_{name}'] = windows[name]
    payload[f'y_endpoint_{name}'] = maps[name].label_endpoint.to_numpy(dtype=np.int8)
    payload[f'y_any_{name}'] = maps[name].label_any.to_numpy(dtype=np.int8)
np.savez_compressed(processed / 'windows.npz', **payload)
mapping = pd.concat(maps.values(), ignore_index=True)
mapping.to_csv(processed / 'window_mapping.csv', index=False)
pd.concat(parts.values(), ignore_index=True).to_csv(processed / 'split_rows.csv', index=False)
scaler_state = {'class': 'StandardScaler', 'features': ['value'], 'mean': scaler.mean_.tolist(),
                'scale': scaler.scale_.tolist(), 'var': scaler.var_.tolist(),
                'n_samples_seen': int(scaler.n_samples_seen_), 'fit_source': NORMAL,
                'fit_source_rows_inclusive': [0, 2303], 'fit_split': 'train',
                'commit': manifest['commit']}
(processed / 'scaler.json').write_text(json.dumps(scaler_state, indent=2), encoding='utf-8')
(ROOT / 'results' / 'experiment_plan.json').write_text(json.dumps(plan, indent=2), encoding='utf-8')

with np.load(processed / 'windows.npz', allow_pickle=False) as reloaded:
    assert set(reloaded.files) == set(payload)
    for key, value in payload.items(): np.testing.assert_array_equal(reloaded[key], value)
read_map = pd.read_csv(processed / 'window_mapping.csv', parse_dates=['timestamp_start', 'timestamp_end'])
pd.testing.assert_frame_equal(read_map, mapping, check_dtype=False)
read_rows = pd.read_csv(processed / 'split_rows.csv', parse_dates=['timestamp'])
pd.testing.assert_frame_equal(read_rows, pd.concat(parts.values(), ignore_index=True), check_dtype=False)
state = json.loads((processed / 'scaler.json').read_text())
np.testing.assert_allclose(state['mean'], scaler.mean_)
np.testing.assert_allclose(state['scale'], scaler.scale_)
for name in parts:
    reconstructed_scaling = (parts[name][['value']].to_numpy()-np.array(state['mean']))/np.array(state['scale'])
    np.testing.assert_allclose(reconstructed_scaling, scaled[name])
checks['split'] = 'PASS: disjoint source-row IDs and chronological development partition'
checks['scaler'] = 'PASS: fitted on train only, manual statistics, held-out transform and inverse verified'
checks['artifact_roundtrip'] = 'PASS: all tensors/labels/CSV rows/mapping/scaler reloaded and compared'
report = {'checks': checks, 'splits': split_summary.to_dict(orient='records'),
          'windows': window_summary.to_dict(orient='records'),
          'scaler': scaler_state, 'trained': False, 'threshold_selected': False,
          'environment': {'sys_executable': sys.executable, 'sys_prefix': sys.prefix, 'sys_base_prefix': sys.base_prefix, 'python': platform.python_version(), 'numpy': np.__version__,
                          'pandas': pd.__version__, 'scikit-learn': sklearn.__version__}}
(ROOT / 'results' / 'preprocessing_checks.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
display(pd.Series(checks))
print('Saved and verified all preprocessing artifacts. STOP before model training.')

train                 PASS: all windows/indices/timestamps/labels/va...
validation            PASS: all windows/indices/timestamps/labels/va...
calibration           PASS: all windows/indices/timestamps/labels/va...
test                  PASS: all windows/indices/timestamps/labels/va...
guards_and_scoring    PASS: mixed boundaries/gaps rejected; syntheti...
split                 PASS: disjoint source-row IDs and chronologica...
scaler                PASS: fitted on train only, manual statistics,...
artifact_roundtrip    PASS: all tensors/labels/CSV rows/mapping/scal...
dtype: object

Saved and verified all preprocessing artifacts. STOP before model training.


## Những điểm cần review trước giai đoạn huấn luyện

1. Chấp nhận cặp NAB nhân tạo làm bộ chính ban đầu, và metric trên khoảng chấm điểm
   thay vì nhãn vật lý chính xác; nếu mục tiêu bắt buộc ECG hoặc đa biến thì đổi bộ
   và kiểm tra lại nhãn/split trước khi huấn luyện.
2. Đối chiếu ảnh yêu cầu chưa nhận được. Xem lại split 8/3/3 ngày, context một ngày,
   mất 287 điểm đầu mỗi tập, endpoint score và ngưỡng đuôi 1% với calibration ngắn.
3. Review cấu hình LSTM và hai baseline. Chỉ chọn checkpoint bằng validation,
   ngưỡng bằng calibration. Không sửa theo F1 test; nếu làm vậy cần test mới.
4. Chưa chạy mô hình, chưa chọn ngưỡng thật, chưa có metric hoặc báo cáo môn học.
   Notebook dừng tại artifact tiền xử lý và kiểm tra; không tự chuyển sang huấn luyện.